<a href="https://colab.research.google.com/github/OjasTamhankar/AML-Lab/blob/main/exp05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load ratings data
ratings = pd.read_csv('/u.data', sep='\t',
                      names=['user_id', 'movie_id', 'rating', 'timestamp'])

# Load movies data
movies = pd.read_csv('/u.item', sep='|', encoding='latin-1',
                     names=['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
                            'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
                            'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
                            'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'])

print(f"Dataset Loaded Successfully!")
print(f"Total Ratings : {len(ratings)}")
print(f"Total Movies  : {len(movies)}")
print(f"Total Users   : {ratings['user_id'].nunique()}\n")

Dataset Loaded Successfully!
Total Ratings : 100000
Total Movies  : 1682
Total Users   : 943



In [20]:
movies

,movie_id,title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,Children,...,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,genres,content
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,Animation Children Comedy,Toy Story (1995) Animation Children Comedy
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,1,0,0,Action Adventure Thriller,GoldenEye (1995) Action Adventure Thriller
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,1,0,0,Thriller,Four Rooms (1995) Thriller
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,Action Comedy Drama,Get Shorty (1995) Action Comedy Drama
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,1,0,0,Crime Drama Thriller,Copycat (1995) Crime Drama Thriller
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Drama,Mat' i syn (1997) Drama
1678,1679,B. Monkey (1998),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,...,0,0,0,1,0,1,0,0,Romance Thriller,B. Monkey (1998) Romance Thriller
1679,1680,Sliding Doors (1998),01-Jan-1998,NaN,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,...,0,0,0,1,0,0,0,0,Drama Romance,Sliding Doors (1998) Drama Romance
1680,1681,You So Crazy (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Comedy,You So Crazy (1994) Comedy


In [11]:
# Create genres string
genre_cols = movies.columns[5:]
movies['genres'] = movies[genre_cols].apply(
    lambda x: ' '.join([col for col, val in zip(genre_cols, x) if val == 1]), axis=1
)

In [12]:
# Combine title and genres
movies['content'] = movies['title'] + ' ' + movies['genres']

In [13]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])

In [14]:
# Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [15]:
# Title to index mapping
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [25]:
def content_based_recommend(title, top_n=5):
    """Recommend similar movies based on content (title + genres)"""
    if title not in indices:
        return f"❌ Movie '{title}' not found!"

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices][['title']].reset_index(drop=True)

print("=== Content-Based Filtering Demo ===")
print(content_based_recommend("GoldenEye (1995)", top_n=5))
print(content_based_recommend("Con Air (1997)", top_n=5))
print(content_based_recommend("B. Monkey (1998)", top_n=5))
print(content_based_recommend("Scream of Stone (Schrei aus Stein) (1991)", top_n=5))

=== Content-Based Filtering Demo ===
                    title
0          Con Air (1997)
1  Fire Down Below (1997)
2        Rock, The (1996)
3       Waterworld (1995)
4          Twister (1996)
                      title
0      Air Force One (1997)
1  Air Up There, The (1994)
2            Air Bud (1997)
3    Fire Down Below (1997)
4           Anaconda (1997)
                        title
0  Spanking the Monkey (1994)
1                 Hush (1998)
2              Tainted (1998)
3       U.S. Marshalls (1998)
4          Little City (1998)
                            title
0                   Scream (1996)
1                 Scream 2 (1997)
2  Sword in the Stone, The (1963)
3               Doors, The (1991)
4           Night on Earth (1991)


In [26]:
# Create User-Item Matrix
user_item_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)

# Convert to sparse matrix
user_item_sparse = csr_matrix(user_item_matrix.values)

# Train KNN model (Item-Based)
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20)
knn.fit(user_item_sparse.T)   # Transpose for item similarity

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=20)

In [28]:
def collaborative_recommend(movie_title, top_n=5):
    """Item-based Collaborative Filtering"""
    if movie_title not in movies['title'].values:
        return f"❌ Movie '{movie_title}' not found!"

    movie_id = movies[movies['title'] == movie_title]['movie_id'].values[0]
    movie_idx = user_item_matrix.columns.get_loc(movie_id)

    distances, indices = knn.kneighbors(
        user_item_matrix.iloc[:, movie_idx].values.reshape(1, -1),
        n_neighbors=top_n + 1
    )

    similar_ids = [user_item_matrix.columns[i] for i in indices.flatten()[1:]]
    recommendations = movies[movies['movie_id'].isin(similar_ids)][['title']].reset_index(drop=True)

    return recommendations

print("\n=== Collaborative Filtering (Item-Based) Demo ===")
print(collaborative_recommend("GoldenEye (1995)", top_n=5))
print(collaborative_recommend("Con Air (1997)", top_n=5))
print(collaborative_recommend("B. Monkey (1998)", top_n=5))
print(collaborative_recommend("Scream of Stone (Schrei aus Stein) (1991)", top_n=5))


=== Collaborative Filtering (Item-Based) Demo ===
                title
0     Stargate (1994)
1      Top Gun (1986)
2  Under Siege (1992)
3    True Lies (1994)
4       Batman (1989)
                                   title
0                       Rock, The (1996)
1  Lost World: Jurassic Park, The (1997)
2                        Face/Off (1997)
3             Mission: Impossible (1996)
4                          Ransom (1996)
                      title
0       Nil By Mouth (1997)
1  Hurricane Streets (1998)
2       Legal Deceit (1997)
3          B. Monkey (1998)
4      Sliding Doors (1998)
                           title
0  Farinelli: il castrato (1994)
1              Stalingrad (1993)
2          Addiction, The (1995)
3        American Buffalo (1996)
4          Romper Stomper (1992)


In [29]:
def hybrid_recommend(user_id, movie_title, top_n=5):
    """Hybrid: Content-based candidates + Collaborative ranking"""
    # Get content-based candidates
    content_df = content_based_recommend(movie_title, top_n=30)
    if isinstance(content_df, str):
        return content_df

    candidate_ids = movies[movies['title'].isin(content_df['title'])]['movie_id'].tolist()

    # Rank candidates using collaborative method
    scores = []
    for mid in candidate_ids:
        try:
            idx = user_item_matrix.columns.get_loc(mid)
            _, indices = knn.kneighbors(
                user_item_matrix.iloc[:, idx].values.reshape(1, -1), n_neighbors=10
            )
            similar_ids = [user_item_matrix.columns[i] for i in indices.flatten()[1:]]
            avg_rating = ratings[ratings['movie_id'].isin(similar_ids)]['rating'].mean()
            scores.append((mid, avg_rating))
        except:
            continue

    # Sort by predicted score
    scores.sort(key=lambda x: x[1], reverse=True)
    top_movie_ids = [x[0] for x in scores[:top_n]]

    return movies[movies['movie_id'].isin(top_movie_ids)][['title']].reset_index(drop=True)

print("\n=== Hybrid Recommendation Demo ===")
print(hybrid_recommend(user_id=196, movie_title="Toy Story (1995)", top_n=5))


=== Hybrid Recommendation Demo ===
                            title
0                     Babe (1995)
1           Lion King, The (1994)
2                  Aladdin (1992)
3           Close Shave, A (1995)
4  Philadelphia Story, The (1940)


In [30]:
print("\n" + "="*60)
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("="*60)
print("1. Content-Based Filtering → Uses movie title + genres")
print("2. Collaborative Filtering → Item-based using KNN (Cosine Similarity)")
print("3. Hybrid Approach → Combines both methods")
print("\nYou can change the movie title and top_n value as per requirement.")


EXPERIMENT COMPLETED SUCCESSFULLY
1. Content-Based Filtering → Uses movie title + genres
2. Collaborative Filtering → Item-based using KNN (Cosine Similarity)
3. Hybrid Approach → Combines both methods

You can change the movie title and top_n value as per requirement.
